In [104]:
print("hello_world")

hello_world


In [105]:
#Loading in dataset
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
df = sns.load_dataset("titanic")
df.head()

df2 = df.copy()

In [106]:
# 1. Check for missing values and handle them.
dfmissingvalues = df.isnull().sum()
print(dfmissingvalues)
# Handle missing values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['deck'] = df['deck'].fillna(df['deck'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
print('Recheck')
# Check again
dfmissingvalues = df.isnull().sum()
print(dfmissingvalues)



survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
Recheck
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64


In [107]:
# 2. Check for duplicate rows and remove them.
print ("Number of Duplicate Rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after removal:", df.duplicated().sum())

Number of Duplicate Rows: 112
Duplicates after removal: 0


In [108]:
# 3. Inspect the "Sex" column for inconsistencies and standardize values
print("Before cleaning:")
print(df['sex'].unique())
print(df['sex'].value_counts(dropna=False))

df['sex'] = (
    df['sex']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'm': 'M',
        'f': 'F',
        'man': 'M',
        'woman': 'F',
        'male' : 'M',
        'female' : 'F'
    })
)

print("\nAfter cleaning:")
print(df['sex'].unique())
print(df['sex'].value_counts(dropna=False))

Before cleaning:
<StringArray>
['male', 'female']
Length: 2, dtype: str
sex
male      486
female    293
Name: count, dtype: int64

After cleaning:
<StringArray>
['M', 'F']
Length: 2, dtype: str
sex
M    486
F    293
Name: count, dtype: int64


In [109]:
# 4. Detect and handle outliers in Age and Fare
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower, upper)

df['age'] = cap_outliers(df['age'])
df['fare'] = cap_outliers(df['fare'])

In [110]:
# 5. Create a new "Family_Size" feature based on "SibSp" and "Parch".
##sibsp siblings & spouses
## parch parents & children
## + ticket holder
df['family_size'] = df['sibsp'] + df['parch'] + 1



In [111]:
# 6. Convert the "Embarked" column to numerical values using label encoding
print(df['embarked'].unique())


le = LabelEncoder()

df['embarked'] = le.fit_transform(df['embarked'])

print(dict(zip(le.classes_, le.transform(le.classes_))))

df.head()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
{'C': np.int64(0), 'Q': np.int64(1), 'S': np.int64(2)}


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size
0,0,3,M,22.0,1,0,7.2500,2,Third,man,True,C,Southampton,no,False,2
1,1,1,F,38.0,1,0,71.2833,0,First,woman,False,C,Cherbourg,yes,False,2
2,1,3,F,26.0,0,0,7.9250,2,Third,woman,False,C,Southampton,yes,True,1
3,1,1,F,35.0,1,0,53.1000,2,First,woman,False,C,Southampton,yes,False,2
4,0,3,M,35.0,0,0,8.0500,2,Third,man,True,C,Southampton,no,True,1


In [116]:
# 7. If a "Ticket_Purchase_Date" column existed, convert it to datetime and extract the year and month.
# Convert to datetime
#df['ticket_purchase_date'] = pd.to_datetime(df['ticket_purchase_date'], errors='coerce')

# Extract year and month
#df['ticket_year'] = df['ticket_purchase_date'].dt.year
#df['ticket_month'] = df['ticket_purchase_date'].dt.month

In [ ]:
# 8. Clean the "Name" column by removing titles and converting to lowercase.
#df['Name'] = (df['Name']
 #             .astype(str)
  #            .str.lower()
   #           .str.replace(r'\b(mr|mrs|miss|ms|dr)\.?\b', '', regex=True)
    #          .str.replace(r'[^a-zA-Z\s]', '', regex=True)
     #         .str.replace(r'\s+', ' ', regex=True)
      #        .str.strip()
#)

In [130]:
# 9. Bin the "Age" column into categories like 'Child', 'Teen', 'Adult', and 'Senior'.
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 12, 19, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'Senior']
)
df.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,age_group
0,0,3,M,22.0,1,0,7.2500,2,Third,man,True,C,Southampton,no,False,2,Adult
1,1,1,F,38.0,1,0,71.2833,0,First,woman,False,C,Cherbourg,yes,False,2,Adult
2,1,3,F,26.0,0,0,7.9250,2,Third,woman,False,C,Southampton,yes,True,1,Adult
3,1,1,F,35.0,1,0,53.1000,2,First,woman,False,C,Southampton,yes,False,2,Adult
4,0,3,M,35.0,0,0,8.0500,2,Third,man,True,C,Southampton,no,True,1,Adult
5,0,3,M,28.0,0,0,8.4583,1,Third,man,True,C,Queenstown,no,True,1,Adult
6,0,1,M,54.0,0,0,51.8625,2,First,man,True,E,Southampton,no,True,1,Adult
7,0,3,M,2.0,3,1,21.0750,2,Third,child,False,C,Southampton,no,False,5,Child
8,1,3,F,27.0,0,2,11.1333,2,Third,woman,False,C,Southampton,yes,False,3,Adult
9,1,2,F,14.0,1,0,30.0708,0,Second,child,False,C,Cherbourg,yes,False,2,Teen
